In [1]:
import sqlite3
import pandas as pd

# === Step 1: 连接数据库 ===
db_path = r"..\query_dataset\stockinfo_query.db"
conn = sqlite3.connect(db_path)

# === Step 2: 查看有哪些表（可选）===
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)
print("📋 数据库中的表：")
print(tables)

# 假设你知道表名，比如叫 "stockinfo"
table_name = tables.iloc[0, 0]  # 默认取第一个表，也可以直接写表名

# === Step 3: 读取表数据 ===
df = pd.read_sql(f"SELECT * FROM [{table_name}]", conn)

# === Step 4: 查看 Symbol 列 ===
if "Symbol" in df.columns:
    symbols = df["Symbol"].dropna().unique()
    print(f"✅ 从表 {table_name} 读取到 {len(symbols)} 个唯一 Symbol：")
    print(symbols[:10])  # 只打印前10个
else:
    print("❌ 表中没有 Symbol 列，请检查表结构。")

# === Step 5: 可选：显示 DataFrame 头部
df.head()


📋 数据库中的表：
        name
0  stockinfo
✅ 从表 stockinfo 读取到 2752 个唯一 Symbol：
['AAAU' 'AADR' 'AAME' 'AAWW' 'AAXJ' 'ABEQ' 'ABMD' 'ACAD' 'ACES' 'ACIO']


,Nasdaq Traded,Symbol,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,NextShares,Company Description
0,Y,AAAU,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Perth Mint Physical Gold ETF offers investors ...
1,Y,AADR,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,AdvisorShares Dorsey Wright ADR ETF offers inv...
2,Y,AAME,Q,G,N,100.0,N,N,N,Atlantic American Corporation provides a range...
3,Y,AAWW,Q,Q,N,100.0,N,N,N,Atlas Air Worldwide Holdings specializes in pr...
4,Y,AAXJ,Q,G,Y,100.0,N,N,N,iShares MSCI All Country Asia ex Japan Index F...


In [3]:
import sqlite3
import pandas as pd

# 你已有的 OpenAI / Azure 客户端和部署名
from openai import AzureOpenAI       # 这里换成你真实的 client 引入
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o"

# === 1. 从数据库获取所有 Symbol ===
db_path = r"..\query_dataset\stockinfo_query.db"
conn = sqlite3.connect(db_path)

table_name = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
).iloc[0, 0]  # 默认第一个表

df = pd.read_sql(f"SELECT * FROM [{table_name}]", conn)
conn.close()

if "Symbol" not in df.columns:
    raise ValueError("表中没有 Symbol 列！")

symbols = df["Symbol"].dropna().unique().tolist()

print(f"✅ 提取到 {len(symbols)} 个唯一 Symbol")

# === 2. 定义 LLM 调用函数 ===
def infer_symbol_for_realreal(symbol_list):
    """
    给 LLM 一组 symbol，请它找出 The RealReal, Inc. 对应的 symbol。
    """
    prompt = (
        f"""You're given a list of stock ticker symbols for publicly traded companies:\n\n{symbols}\n\n
Your task is to figure out which ticker symbol belongs to **The RealReal, Inc.**.
Output only the ticker symbol itself, no extra text. If none of them corresponds to The RealReal, Inc., reply with 'None'.
"""
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a financial assistant who maps company names to their stock symbols."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=10
        )
        output = response.choices[0].message.content.strip()
        return output
    except Exception as e:
        print(f"[ERROR] LLM call failed: {e}")
        return None

# === 3. 调用函数 ===
realreal_symbol = infer_symbol_for_realreal(symbols)
print(f"🎯 The RealReal, Inc. 对应的股票代码是: {realreal_symbol}")


✅ 提取到 2752 个唯一 Symbol
🎯 The RealReal, Inc. 对应的股票代码是: REAL


In [4]:
import duckdb
import pandas as pd

# === 1. 连接 DuckDB ===
db_path = r"..\query_dataset\stocktrade_query.db"
con = duckdb.connect(database=db_path, read_only=True)

# === 2. 查看有哪些表（可选检查）===
tables = con.execute("SHOW TABLES").fetchdf()
print("📋 数据库中的表：")
print(tables)

if realreal_symbol not in tables["name"].values:
    raise ValueError(f"❌ 没有找到表：{realreal_symbol}，请检查股票代码是否正确！")

# === 3. 查询目标表中 2020 年的最大 Adj Close ===
query = f"""
SELECT MAX("Adj Close") AS max_adj_close_2020
FROM "{realreal_symbol}"
WHERE DATE >= '2020-01-01' AND DATE <= '2020-12-31'
"""

result_df = con.execute(query).fetchdf()
max_adj_close = result_df.iloc[0, 0]

print(f"🎯 {realreal_symbol} 在 2020 年的最大 Adjusted Close 价格是: {max_adj_close}")


📋 数据库中的表：
      name
0     AAAU
1     AADR
2     AAME
3     AAWW
4     AAXJ
...    ...
2748  ZMLP
2749   ZNH
2750  ZROZ
2751   ZSL
2752   ZTR

[2753 rows x 1 columns]
🎯 REAL 在 2020 年的最大 Adjusted Close 价格是: 18.440000534057617


In [9]:
from openai import AzureOpenAI 
import sqlite3
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm      # 这里换成你真实的 client 引入
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o"

import sqlite3
import pandas as pd

# Step 1: get ETF=Y rows
sqlite_path = r"..\query_dataset\stockinfo_query.db"

def get_etf_rows(sqlite_path):
    conn = sqlite3.connect(sqlite_path)
    table_name = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    ).iloc[0, 0]
    df = pd.read_sql(f"SELECT * FROM [{table_name}] WHERE ETF = 'Y'", conn)
    conn.close()
    return df

df_etf = get_etf_rows(sqlite_path)
print(f"✅ Found {len(df_etf)} ETF rows.")

def ask_llm_if_nyse_arca(listing_exchange_code):
    """
    Ask LLM whether this listing exchange code corresponds to NYSE Arca.
    Return True if Yes, False otherwise.
    """
    prompt = f"""
Here is the mapping rule for exchanges:
A = NYSE MKT
N = New York Stock Exchange (NYSE)
P = NYSE ARCA
Z = BATS Global Markets (BATS)
V = Investors' Exchange, LLC (IEXG)
Q = NASDAQ Global Select Market (top-tier NASDAQ market)

Given a security with Listing Exchange code: `{listing_exchange_code}`  
Does this correspond to NYSE Arca (P)?  
Reply only with `Yes` or `No`.
"""
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a financial assistant who maps exchange codes to their full names and checks if it is NYSE Arca."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )
        output = response.choices[0].message.content.strip().lower()
        return output.startswith("yes")
    except Exception as e:
        print(f"[ERROR] LLM call failed for code {listing_exchange_code}: {e}")
        return None

def process_batch(batch_indices, df):
    """
    Process one batch of rows by submitting each to LLM in parallel.
    """
    batch_results = {}
    with ThreadPoolExecutor(max_workers=len(batch_indices)) as executor:
        futures = {
            executor.submit(ask_llm_if_nyse_arca, str(df.iloc[idx]["Listing Exchange"]).strip()): idx
            for idx in batch_indices
        }
        for future in as_completed(futures):
            idx = futures[future]
            try:
                res = future.result()
                batch_results[idx] = res
            except Exception as e:
                print(f"❌ Row {idx} failed during batch: {e}")
                batch_results[idx] = None
    return batch_results


def parallel_check_with_batches(df, batch_size=50):
    """
    Run the checks in batches. Retry failed batches at the end.
    """
    total_rows = len(df)
    indices = list(range(total_rows))
    batches = [indices[i:i+batch_size] for i in range(0, total_rows, batch_size)]

    results = [None] * total_rows
    failed_batches = []

    print(f"🚀 Processing {len(batches)} batches of up to {batch_size} each...")

    for i, batch_indices in enumerate(tqdm(batches, desc="Main batches")):
        batch_results = process_batch(batch_indices, df)
        for idx, res in batch_results.items():
            if res is None:
                failed_batches.append(idx)
            results[idx] = res

    if failed_batches:
        print(f"\n🔁 Retrying {len(failed_batches)} failed rows...")
        retry_results = process_batch(failed_batches, df)
        for idx, res in retry_results.items():
            results[idx] = res if res is not None else False

    return results

yes_no_flags = parallel_check_with_batches(df_etf, batch_size=50)

df_etf["is_nyse_arca"] = [bool(flag) for flag in yes_no_flags]
filtered_df = df_etf[df_etf["is_nyse_arca"]].copy()

print(f"✅ Found {len(filtered_df)} ETF rows listed on NYSE Arca.")
filtered_df.head()


✅ Found 2165 ETF rows.
🚀 Processing 44 batches of up to 50 each...


Main batches: 100%|██████████| 44/44 [00:36<00:00,  1.19it/s]

✅ Found 1435 ETF rows listed on NYSE Arca.


,Nasdaq Traded,Symbol,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,NextShares,Company Description,is_nyse_arca
0,Y,AAAU,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Perth Mint Physical Gold ETF offers investors ...,True
1,Y,AADR,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,AdvisorShares Dorsey Wright ADR ETF offers inv...,True
3,Y,ABEQ,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Absolute Core Strategy ETF is an investment fu...,True
6,Y,ACSG,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Xtrackers MSCI ACWI ex USA ESG Leaders Equity ...,True
9,Y,ACWF,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,iShares Edge MSCI Multifactor Global ETF is an...,True


In [10]:
import duckdb

duckdb_path = r"..\query_dataset\stocktrade_query.db"
con = duckdb.connect(duckdb_path, read_only=True)
def had_adj_close_above_200_2015(symbol):
    """
    Check if the given symbol's table has any row in 2015 with Adj Close > 200.
    """
    try:
        query = f"""
        SELECT 1
        FROM "{symbol}"
        WHERE DATE >= '2015-01-01' AND DATE <= '2015-12-31'
          AND "Adj Close" > 200
        LIMIT 1
        """
        result = con.execute(query).fetchone()
        return result is not None
    except Exception as e:
        print(f"[ERROR] Checking symbol {symbol}: {e}")
        return False
qualified_symbols = []

for symbol in tqdm(filtered_df["Symbol"], desc="Checking Adj Close > 200 in 2015"):
    if had_adj_close_above_200_2015(symbol):
        qualified_symbols.append(symbol)

print(f"✅ Found {len(qualified_symbols)} symbols with Adj Close > 200 in 2015.")
final_df = filtered_df[filtered_df["Symbol"].isin(qualified_symbols)].copy()
print(f"📄 Final dataset has {len(final_df)} rows.")
final_df.head()


Checking Adj Close > 200 in 2015: 100%|██████████| 1435/1435 [00:00<00:00, 4500.16it/s]

✅ Found 31 symbols with Adj Close > 200 in 2015.
📄 Final dataset has 31 rows.


,Nasdaq Traded,Symbol,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,NextShares,Company Description,is_nyse_arca
119,Y,BOIL,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,ProShares Ultra Bloomberg Natural Gas offers i...,True
171,Y,BZQ,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,ProShares UltraShort MSCI Brazil Capped offers...,True
224,Y,COM,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Auspice Broad Commodity Strategy ETF ...,True
347,Y,DUST,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Daily Gold Miners Index Bear 2X Share...,True
394,Y,EDZ,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Emerging Markets Bear 3X Shares provi...,True


In [11]:
def extract_company_name(description):
    """
    Ask LLM to extract the company name from the description.
    """
    prompt = f"""
You are given the following company description:
\"\"\"{description}\"\"\"

Your task is to extract the **official company name** from this description.
Output only the company name, nothing else.
"""
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a financial assistant who extracts company names from descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=20
        )
        output = response.choices[0].message.content.strip()
        return output
    except Exception as e:
        print(f"[ERROR] LLM call failed for description: {e}")
        return None
company_names = []

for desc in tqdm(final_df["Company Description"], desc="Extracting company names"):
    name = extract_company_name(desc)
    company_names.append(name)
final_df["Company Name"] = company_names
final_df.head()


Extracting company names: 100%|██████████| 31/31 [00:13<00:00,  2.32it/s]


,Nasdaq Traded,Symbol,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,NextShares,Company Description,is_nyse_arca,Company Name
119,Y,BOIL,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,ProShares Ultra Bloomberg Natural Gas offers i...,True,ProShares Ultra Bloomberg Natural Gas
171,Y,BZQ,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,ProShares UltraShort MSCI Brazil Capped offers...,True,ProShares UltraShort MSCI Brazil Capped
224,Y,COM,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Auspice Broad Commodity Strategy ETF ...,True,Direxion Auspice Broad Commodity Strategy ETF
347,Y,DUST,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Daily Gold Miners Index Bear 2X Share...,True,Direxion Daily Gold Miners Index Bear 2X Shares
394,Y,EDZ,P,Not applicable or not NASDAQ-listed,Y,100.0,N,None,N,Direxion Emerging Markets Bear 3X Shares provi...,True,Direxion Emerging Markets Bear 3X Shares
